In [ ]:
import datetime
import glob
import gc

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl

from scipy import interpolate

import cartopy.crs as ccrs
import cartopy.feature

import parcels

import load_copernics_fieldset
from kernels import WindageRK2, Stranding, LoggerheadParticle, StokesDriftRK2, UnbeachingBySampling

In [ ]:
df_in = pd.read_excel("stranding_locations.xlsx")
df_in

In [ ]:
offsets = {  # TODO check if still needed with UnbeachingBySampling kernel
    "CC26": (0.02, 0),
    "CC32": (0, -0.02),
    "CC23": (0.015, 0),
    "CC33": (0.01, 0),
    "CC31": (0, -0.02),
}

In [ ]:
N = 10

def run_sim(beach_date, beach_lat, beach_lon, beach_id):
    fieldset = load_copernics_fieldset.create_fieldset(startdate=beach_date-datetime.timedelta(days=14), enddate=beach_date)

    # Windage coefficient [fraction]
    fieldset.wind_coeff = 0.01

    lons = beach_lon + np.random.uniform(-0.01, 0.01, size=N)
    lats = beach_lat + np.random.uniform(-0.01, 0.01, size=N)

    pfile = parcels.ParticleFile(
        f"Loggerhead_Simulation_{beach_id}.parquet",
        outputdt=np.timedelta64(1, 'h'),
        mode="w",
    )
    # fieldset.output_file = pfile  # for writing stranded particles

    pset = parcels.ParticleSet(
        fieldset=fieldset,
        pclass=LoggerheadParticle,
        lon=lons,
        lat=lats,
        time=np.datetime64(beach_date),
    )

    kernels = [
        # Stranding,
        parcels.kernels.AdvectionRK2,
        # WindageRK2,
        # StokesDriftRK2,
        UnbeachingBySampling,
    ]

    pset.execute(
        kernels=kernels,
        runtime=np.timedelta64(14, 'D'),
        dt=-np.timedelta64(10, 'm'),
        output_file=pfile,
    )
    gc.collect()

for index, row in df_in.iterrows():
    location = row["Locatie"]
    beach_id = row["ID"]
    beach_date = row["Datum"]
    if beach_date < datetime.datetime(2024, 4, 1):
        print(f"Invalid date for location {beach_id}: {beach_date}")
        continue
    if pd.isna(location):
        print(f"Invalid location for location {beach_id}: {location}")
        continue
    offset = (0, 0) if beach_id not in offsets.keys() else offsets[beach_id]
    beach_lat = float(location.split(",")[0].strip()) + offset[0]
    beach_lon = float(location.split(",")[1].strip()) + offset[1]
    print(f"Stranding location {beach_id}: Latitude: {beach_lat}, Longitude: {beach_lon} on {beach_date}")

    run_sim(beach_date, beach_lat, beach_lon, beach_id)


In [ ]:
files = sorted(glob.glob("Loggerhead_Simulation_C*.parquet"))

beach_date = datetime.datetime(2024, 4, 24)
fieldset = load_copernics_fieldset.create_fieldset(startdate=beach_date - datetime.timedelta(days=14), enddate=beach_date)
land_mask_condition = (fieldset.U.data.isel(time=0, depth=0) == 0) & \
                      (fieldset.V.data.isel(time=0, depth=0) == 0)
landmask = np.ma.masked_where(land_mask_condition, land_mask_condition)

lon, lat = fieldset.U.grid.lon, fieldset.U.grid.lat
lon_centers = lon[1:] - np.diff(lon) / 2
lat_centers = lat[1:] - np.diff(lat) / 2
x, y = np.meshgrid(lon_centers, lat_centers)

fl = interpolate.RectBivariateSpline(lat, lon, landmask.mask, kx=1, ky=1)
lmask = (fl(y[:, 0], x[0, :]) > 0.95).astype(int)

for file in files:
    df = parcels.read_particlefile(file)

    fig = plt.figure(figsize = (10, 8))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_facecolor('aliceblue')
    ax.gridlines(draw_labels=['left','bottom'], zorder=2, alpha=0.3, linestyle='--')

    ax.pcolormesh(x, y, lmask)

    for traj in df.partition_by("particle_id", maintain_order=True):
        ax.plot(traj["lon"], traj["lat"], "-", zorder=3)
        ax.plot(traj["lon"][0], traj["lat"][0], "ko", zorder=4)
        ax.plot(traj["lon"][-1], traj["lat"][-1], "ro", zorder=4)

    ax.set_xlim([df["lon"].min() - 0.1, df["lon"].max() + 0.1])
    ax.set_ylim([df["lat"].min() - 0.1, df["lat"].max() + 0.1])
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.title('Loggerhead Particle Trajectories for ' + file.split("_")[-1].split(".")[0])
    plt.show()
